# 02c Estimate: the regression models with indicators (model family 2)

**Purpose:** The second model family: the three regressions of `02b_estimate_regression.ipynb` (linear regression, Ridge, Lasso) with indicators next to the series' own past, in the same backtest and with the same rules, so that any gain or loss can be attributed to the indicators (project plan: one block at a time). The indicators of `01b` enter **one at a time** (the routes, the neighbouring countries) and **both together** ("all indicators"); comparing these shows which data lower the error.
**Input:** `../data/processed/asylum_clean.csv` (from `01_preprocess.ipynb`), `../data/processed/routes_clean.csv` and `../data/processed/neighbours_clean.csv` (from `01b_preprocess_indicators.ipynb`).
**Output:** `../data/processed/backtest_forecasts_indicator.csv` (same columns as `backtest_forecasts_regression.csv`) and `../output/tables/02c_lasso_kept.csv`.

As before, the test period (2024-10 to 2026-09) is **not used**. We compare the models in `03_evaluate.ipynb`.

In [1]:
import numpy as np
import pandas as pd
import warnings

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso

np.random.seed(42)

## 1. Load the target and the indicators

We put the log of each indicator next to the target, month by month. `reindex` puts each indicator on the months of the target; months an indicator does not cover stay `NaN` (the routes start in January 2009, the target in January 2008).

*Not seen in class:* `reindex(months)` puts a series on a given list of row labels and leaves `NaN` where a label is missing.

In [2]:
clean = pd.read_csv("../data/processed/asylum_clean.csv")
clean["month"] = pd.to_datetime(clean["month"])
clean = clean.set_index("month")
y = clean["applicants_total"].to_numpy(dtype=float)
log_y = clean["log_applicants"].to_numpy(dtype=float)
months = clean.index

def load(path):
    data = pd.read_csv(path)
    data["month"] = pd.to_datetime(data["month"])
    return data.set_index("month")

log_routes = load("../data/processed/routes_clean.csv")["log_routes"].reindex(months).to_numpy(dtype=float)
log_neighbours = load("../data/processed/neighbours_clean.csv")["log_neighbours"].reindex(months).to_numpy(dtype=float)

FIRST_ORIGIN = pd.Timestamp("2012-12-01")
LAST_BACKTEST = pd.Timestamp("2024-09-01")
H_MAX = 3
first_origin = int(np.where(months == FIRST_ORIGIN)[0][0])
last_backtest = int(np.where(months == LAST_BACKTEST)[0][0])
print("Missing months: routes", np.isnan(log_routes).sum(), "| neighbours", np.isnan(log_neighbours).sum())

Missing months: routes 12 | neighbours 1


## 2. The indicator features

`01b` fixed the rule: **at a forecast origin $t$, the newest month we may use is $t-1$** for both indicators (both are published about one month later than the target). We decided on two features per indicator before running any model, with $R$ = detections on the three routes and $N$ = applications in DE + AT + FR + IT:

| Feature | Formula | Idea |
|---|---|---|
| `routes_change_last_month` | $\log R_{t-1} - \log R_{t-2}$ | the newest monthly change on the routes |
| `routes_change_3_months` | $\log R_{t-1} - \log R_{t-4}$ | the change over the last three known months |
| `neighbours_change_last_month` | $\log N_{t-1} - \log N_{t-2}$ | the newest monthly change in the neighbouring countries |
| `neighbours_change_3_months` | $\log N_{t-1} - \log N_{t-4}$ | the change over the last three known months |

They are **changes**, like the other features, for the reason of notebook 200 (trending levels can look related without any link). We did not use a change over a whole year: it would need the indicator 13 months back, and the first training windows would become very small.

Three combinations: the routes alone, the neighbours alone, and both ("all indicators"). All other features, the target, the models, the grids of `alpha` and the validation inside each training window are **exactly those of `02b_estimate_regression.ipynb`**.

In [3]:
features = {
    "routes": {
        "routes_change_last_month": lambda t: log_routes[t - 1] - log_routes[t - 2],
        "routes_change_3_months": lambda t: log_routes[t - 1] - log_routes[t - 4],
    },
    "neighbours": {
        "neighbours_change_last_month": lambda t: log_neighbours[t - 1] - log_neighbours[t - 2],
        "neighbours_change_3_months": lambda t: log_neighbours[t - 1] - log_neighbours[t - 4],
    },
}
indicators = list(features)
combinations = {"routes": ["routes"], "neighbours": ["neighbours"], "all indicators": indicators}

def make_table(h, combination):
    # Same table as in 02b, plus the features of the indicators (months up to t - 1 only)
    extra_features = {k: f for b in combinations[combination] for k, f in features[b].items()}
    rows = []
    for t in range(12, len(log_y) - h):
        values = {name: f(t) for name, f in extra_features.items()}
        if any(np.isnan(v) for v in values.values()):     # an indicator starts when all its months exist
            continue
        rows.append({
            "t": t,
            "change_last_month": log_y[t] - log_y[t - 1],
            "change_month_before": log_y[t - 1] - log_y[t - 2],
            "same_change_last_year": log_y[t + h - 12] - log_y[t - 12],
            **values,
            "calendar_month": months[t + h].month,
            "target": log_y[t + h] - log_y[t],
        })
    table = pd.DataFrame(rows).set_index("t")
    dummies = pd.get_dummies(table["calendar_month"], prefix="month", drop_first=True).astype(float)
    names = ["change_last_month", "change_month_before", "same_change_last_year"] + list(extra_features)
    X = pd.concat([table[names], dummies], axis=1)
    return X, table["target"]

tables = {(c, h): make_table(h, c) for c in combinations for h in range(1, H_MAX + 1)}
for c in combinations:
    X2, _ = tables[(c, 2)]
    print(f"{c:15s} h = 2: {X2.shape[0]} rows, {X2.shape[1]} features, first origin {months[X2.index.min()]:%Y-%m}, "
          f"missing values {X2.isna().sum().sum()}")
tables[("all indicators", 2)][0][[k for b in indicators for k in features[b]]].describe().round(3)

routes          h = 2: 207 rows, 16 features, first origin 2009-05, missing values 0
neighbours      h = 2: 211 rows, 16 features, first origin 2009-01, missing values 0
all indicators  h = 2: 207 rows, 18 features, first origin 2009-05, missing values 0


,routes_change_last_month,routes_change_3_months,neighbours_change_last_month,neighbours_change_3_months
count,207.000,207.000,207.000,207.000
mean,0.002,0.006,0.005,0.017
std,0.376,0.716,0.161,0.260
min,-1.645,-2.216,-0.919,-1.560
25%,-0.229,-0.475,-0.084,-0.121
50%,0.039,0.093,0.007,0.015
75%,0.218,0.479,0.114,0.159
max,1.227,2.384,0.477,1.089


**What we see.** For $h = 2$, the tables with one indicator have 16 features (the three own changes, two indicator features, 11 month dummies), the table with both indicators 18. No value is missing. The tables with the routes start at the origin May 2009 (the routes start in January 2009 and the 3-month change needs $t-4$), the table with the neighbours alone in January 2009, like `02b`. The route changes move much more than the neighbour changes: standard deviation 0.38 against 0.16 for the monthly change.

## 3. Models, grids and the choice of `alpha`: the same as in `02b`

The functions below are copied from `02b_estimate_regression.ipynb` without any change (scaling inside the pipeline, two validation blocks of 12 months inside each training window, the same grids).

In [4]:
def make_model(name, alpha=None):
    if name == "Linear regression":
        return make_pipeline(StandardScaler(), LinearRegression())
    if name == "Ridge":
        return make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    return make_pipeline(StandardScaler(), Lasso(alpha=alpha, max_iter=10000))

alpha_grids = {
    "Linear regression": [None],
    "Ridge": [0.1, 1, 10, 100, 1000, 10000],
    "Lasso": [0.0001, 0.001, 0.003, 0.01, 0.03, 0.1, 0.3],
}

def choose_alpha(name, X, target, T, h, n_blocks=2, block=12):
    grid = alpha_grids[name]
    if len(grid) == 1:
        return grid[0]
    first_row, last_row = X.index.min(), T - h
    scores = []
    for alpha in grid:
        mse_blocks = []
        for b in range(n_blocks):
            start = last_row - n_blocks * block + 1 + b * block
            valid_rows = list(range(start, start + block))
            train_rows = list(range(first_row, start - h + 1))
            model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
            errors = target.loc[valid_rows].to_numpy() - model.predict(X.loc[valid_rows])
            mse_blocks.append(np.mean(errors ** 2))
        scores.append(np.mean(mse_blocks))
    return grid[int(np.argmin(scores))]

## 4. The backtest loop

The same loop as in `02b`, once per combination: the same 141 origins and horizons. We check again that no training target lies after the origin (`assert`). This takes about four minutes.

In [5]:
rows = []
for combination in combinations:
    for t in range(first_origin, last_backtest):
        for h in range(1, H_MAX + 1):
            target_pos = t + h
            if target_pos > last_backtest:
                continue
            X, target = tables[(combination, h)]
            train_rows = list(range(X.index.min(), t - h + 1))
            assert max(train_rows) + h <= t                    # no training target after the origin
            for name in alpha_grids:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    alpha = choose_alpha(name, X, target, t, h)
                    model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
                forecast_log = log_y[t] + model.predict(X.loc[[t]])[0]
                rows.append({
                    "model": f"{name} + {combination}", "origin": months[t], "month": months[target_pos],
                    "h": h, "forecast_log": forecast_log, "actual_log": log_y[target_pos],
                    "forecast": np.exp(forecast_log), "actual": y[target_pos], "alpha": alpha,
                })

forecasts = pd.DataFrame(rows)
print("Number of stored forecasts:", forecasts.shape[0])

Number of stored forecasts: 3780


## 5. Checks on the stored forecasts

In [6]:
print(forecasts.groupby(["model", "h"]).size().unstack())
print()
print("All forecast months in the backtest (before 2024-10):", (forecasts["month"] < pd.Timestamp("2024-10-01")).all())
print("All forecast months after their origin:", (forecasts["month"] > forecasts["origin"]).all())
print("Missing or infinite forecasts:", (~np.isfinite(forecasts["forecast_log"])).sum())

h                                     1    2    3
model                                            
Lasso + all indicators              141  140  139
Lasso + neighbours                  141  140  139
Lasso + routes                      141  140  139
Linear regression + all indicators  141  140  139
Linear regression + neighbours      141  140  139
Linear regression + routes          141  140  139
Ridge + all indicators              141  140  139
Ridge + neighbours                  141  140  139
Ridge + routes                      141  140  139

All forecast months in the backtest (before 2024-10): True
All forecast months after their origin: True
Missing or infinite forecasts: 0


**What we see.** As in `02b`: each of the nine models has 141, 140 and 139 forecasts for $h = 1, 2, 3$, 3,780 together, all in the backtest, all after their origin, none missing. With the `assert` in the loop, this shows that the indicators used only months known at each origin.

### Which penalties were chosen?

As in `02b` (notebook 209), we check that the chosen `alpha` does not sit mostly at an edge of the grid.

In [7]:
for name in ["Ridge", "Lasso"]:
    chosen = forecasts[forecasts["model"].str.startswith(name + " +")]
    print(f"{name}: how often each alpha was chosen")
    print(chosen.groupby(["model", "alpha"]).size().unstack(fill_value=0).to_string())
    print()

Ridge: how often each alpha was chosen
alpha                   0.1      1.0      10.0     100.0    1000.0   10000.0
model                                                                       
Ridge + all indicators       68       17       63      195       59       18
Ridge + neighbours           63       25       72      174       56       30
Ridge + routes               92        9       50      196       65        8

Lasso: how often each alpha was chosen
alpha                   0.0001  0.0010  0.0030  0.0100  0.0300  0.1000  0.3000
model                                                                         
Lasso + all indicators      53      22      55      85      96      63      46
Lasso + neighbours          55      28      43      92      99      74      29
Lasso + routes             100      11      22      93      78      66      50



**What we see.** Ridge most often chose `alpha = 100` (174 to 196 of 420 times); the largest value (10,000) was chosen at most 30 times. Lasso most often chose 0.01 or 0.03; with the routes alone it chose the smallest value (0.0001, almost no penalty) 100 times.

**What this means for us.** The chosen values lie mostly inside the grids, so the grids are wide enough.

## 6. What do the models do with the indicators?

The coefficients for $h = 2$ at the last origin (August 2024), on standardised features (notebook 105), for the models with all indicators. A coefficient says what helps to *predict*, not what *causes*.

In [8]:
t = last_backtest - 1
h = 2
X, target = tables[("all indicators", h)]
train_rows = list(range(X.index.min(), t - h + 1))

coefficients = pd.DataFrame(index=X.columns)
for name in alpha_grids:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha(name, X, target, t, h)
        model = make_model(name, alpha).fit(X.loc[train_rows], target.loc[train_rows])
    coefficients[f"{name} (alpha = {alpha})"] = model[-1].coef_

print("Origin:", months[t], "| horizon h =", h, "| training rows:", len(train_rows))
coefficients.iloc[:7].round(3)

Origin: 2024-08-01 00:00:00 | horizon h = 2 | training rows: 182


,Linear regression (alpha = None),Ridge (alpha = 100),Lasso (alpha = 0.0001)
change_last_month,-0.018,-0.008,-0.018
change_month_before,-0.011,-0.007,-0.011
same_change_last_year,0.005,0.021,0.005
routes_change_last_month,0.014,0.011,0.014
routes_change_3_months,-0.028,-0.005,-0.028
neighbours_change_last_month,0.007,-0.008,0.006
neighbours_change_3_months,-0.050,-0.030,-0.050


Lasso can set a coefficient to exactly 0 (notebook 105). At every origin of the backtest ($h = 2$, all indicators), we record which indicator features it keeps: a feature kept at most origins carries information that the series' own past does not.

In [9]:
indicator_names = [k for b in indicators for k in features[b]]
kept = []
for t in range(first_origin, last_backtest - 1):          # all origins with a known h = 2 target
    X, target = tables[("all indicators", 2)]
    train_rows = list(range(X.index.min(), t - 2 + 1))
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        alpha = choose_alpha("Lasso", X, target, t, 2)
        model = make_model("Lasso", alpha).fit(X.loc[train_rows], target.loc[train_rows])
    kept.append(pd.Series(model[-1].coef_, index=X.columns)[indicator_names])
kept = pd.DataFrame(kept)
lasso_kept = pd.DataFrame({"indicator": [next(b for b in indicators if k in features[b]) for k in indicator_names],
                           "share of origins kept": (kept != 0).mean().round(2),
                           "mean coefficient": kept.mean().round(3)})
lasso_kept.index.name = "feature"
lasso_kept.to_csv("../output/tables/02c_lasso_kept.csv")
print("Origins:", len(kept))
lasso_kept

Origins: 140


,indicator,share of origins kept,mean coefficient
feature,,,
routes_change_last_month,routes,0.74,0.010
routes_change_3_months,routes,0.36,-0.009
neighbours_change_last_month,neighbours,0.34,0.003
neighbours_change_3_months,neighbours,0.46,-0.019


**What we see.**
- At the last origin (182 training rows), Ridge chose a strong penalty (`alpha = 100`) and keeps every indicator coefficient small (at most 0.03 in absolute value). In all three models the largest indicator coefficient is the 3-month change of the neighbours (−0.030 to −0.050). The two features of the same indicator have opposite signs, so no sign should be read on its own.
- Over all 140 origins, Lasso keeps the **monthly route change** at 74% of the origins; the other three indicator features at only 34% to 46%.

**What this means for us.** Only the newest route month is kept at most origins. Whether the indicators lower the *forecast errors* is measured in `03_evaluate.ipynb` (section 7).

## 7. Save the forecasts

In [10]:
forecasts.to_csv("../data/processed/backtest_forecasts_indicator.csv", index=False)
print("Saved", forecasts.shape[0], "forecasts with the columns", list(forecasts.columns))

Saved 3780 forecasts with the columns ['model', 'origin', 'month', 'h', 'forecast_log', 'actual_log', 'forecast', 'actual', 'alpha']


## Summary

- **Models:** the three regressions of `02b` with the same features, target, grids and validation, plus two features per indicator: the routes alone, the neighbouring countries alone, and both together ("all indicators"). All indicator months are known at the cutoff ($t-1$).
- **Checks:** 3,780 forecasts, all in the backtest, after their origin, none missing; the chosen `alpha` lies mostly inside the grids.
- **What the models do:** Ridge keeps the indicator coefficients small; Lasso keeps only the monthly route change at most origins (74%).
- **Output:** `../data/processed/backtest_forecasts_indicator.csv` and `../output/tables/02c_lasso_kept.csv`, compared in `03_evaluate.ipynb`.